# Práctica 5 · Reto: el entrenador personal
Deep Learning para Visión Artificial · Sesión 5 · **20 minutos**

Vas a convertir los puntos de MediaPipe en **decisiones**. Completa los huecos `# TODO`; haz los niveles en orden.

| Nivel | Qué hay que conseguir |
|---|---|
| 1 | Decir si una persona tiene **las dos manos por encima de la cabeza** |
| 2 | Clasificar cada brazo: **abajo, en cruz o arriba** según el ángulo del hombro |
| 3 | **Contar sentadillas** en un vídeo con una máquina de estados |

Recuerda: `y` crece **hacia abajo** en la imagen. Una mano «más arriba» tiene una `y` más pequeña.

## Preparación (ejecuta sin cambiar nada)

In [ ]:
# Funciona igual en Colab y en VS Code / Jupyter en tu ordenador
import os, sys, subprocess, urllib.request
try:
    import google.colab
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

def instalar(*paquetes):
    """pip install en el mismo Python que ejecuta este cuaderno."""
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *paquetes], check=True)

def falta(modulo):
    return subprocess.run([sys.executable, '-c', f'import {modulo}'], capture_output=True).returncode != 0

def descargar(url, fichero):
    if not os.path.exists(fichero):
        urllib.request.urlretrieve(url, fichero)

if falta('mediapipe'):
    print('Instalando MediaPipe (tarda un poco)...')
    instalar('mediapipe')
# Colab trae opencv-python 5, que tapa al OpenCV con contrib de MediaPipe y no tiene Haar (Demo 7).
# Si falta CascadeClassifier, dejamos un único OpenCV: opencv-contrib-python (el que pide MediaPipe).
comprobar = 'import cv2, sys; sys.exit(0 if hasattr(cv2, "CascadeClassifier") else 1)'
if subprocess.run([sys.executable, '-c', comprobar], capture_output=True).returncode != 0:
    print('Dejo un único OpenCV con contrib (lo usan MediaPipe y Haar)...')
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'opencv-python', 'opencv-python-headless',
                    'opencv-contrib-python', 'opencv-contrib-python-headless'])
    instalar('opencv-contrib-python')
    if 'cv2' in sys.modules:
        raise SystemExit('Instalado. Reinicia el kernel (Colab: Entorno de ejecución → Reiniciar sesión; '
                         'VS Code: botón Restart) y vuelve a ejecutar esta celda.')
import math, time
import cv2
import numpy as np
import matplotlib.pyplot as plt
import mediapipe as mp
assert hasattr(cv2, 'CascadeClassifier'), 'Reinicia el kernel y vuelve a ejecutar la Demo 0.'
from mediapipe.tasks import python as mpt
from mediapipe.tasks.python import vision

URL = 'https://storage.googleapis.com/mediapipe-models'
descargar(f'{URL}/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task', 'pose.task')
descargar(f'{URL}/gesture_recognizer/gesture_recognizer/float16/latest/gesture_recognizer.task', 'gestos.task')
descargar(f'{URL}/face_landmarker/face_landmarker/float16/latest/face_landmarker.task', 'cara.task')
descargar('https://github.com/ultralytics/assets/releases/download/v0.0.0/bus.jpg', 'calle.jpg')
print('MediaPipe', mp.__version__, '· Colab' if EN_COLAB else '· local')

P = vision.PoseLandmark                   # P.NOSE, P.LEFT_ELBOW, P.RIGHT_KNEE...
CONEXIONES_POSE = [(c.start, c.end) for c in vision.PoseLandmarksConnections.POSE_LANDMARKS]
CONEXIONES_MANO = [(c.start, c.end) for c in vision.HandLandmarksConnections.HAND_CONNECTIONS]

def mostrar(img, titulo='', ancho=8):
    plt.figure(figsize=(ancho, ancho * img.shape[0] / img.shape[1]))
    plt.imshow(img); plt.title(titulo); plt.axis('off'); plt.show()

def a_mp(img):
    return mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(img))

def dibujar_esqueleto(imagen, personas, conexiones, grosor=4):
    salida = imagen.copy(); H, W = imagen.shape[:2]
    px = lambda p: (int(p.x * W), int(p.y * H))
    for puntos in personas:
        for a, b in conexiones:
            cv2.line(salida, px(puntos[a]), px(puntos[b]), (217, 105, 46), grosor, cv2.LINE_AA)
        for p in puntos:
            cv2.circle(salida, px(p), grosor + 2, (30, 42, 54), -1, cv2.LINE_AA)
    return salida

def angulo(a, b, c, W=1, H=1):
    """Ángulo en b (grados) formado por a-b-c. W y H pasan de coordenadas 0-1 a píxeles."""
    v1 = ((a.x - b.x) * W, (a.y - b.y) * H)
    v2 = ((c.x - b.x) * W, (c.y - b.y) * H)
    coseno = (v1[0] * v2[0] + v1[1] * v2[1]) / (math.hypot(*v1) * math.hypot(*v2) + 1e-9)
    return math.degrees(math.acos(max(-1, min(1, coseno))))

In [ ]:
def foto_webcam(fichero='webcam.jpg'):
    """Colab: botón «Hacer foto» en el navegador. VS Code / Jupyter local: la webcam del ordenador."""
    if EN_COLAB:
        from IPython.display import display, Javascript
        from google.colab.output import eval_js
        from base64 import b64decode
        display(Javascript("""
        async function hacerFoto() {
          const div = document.createElement('div');
          const boton = document.createElement('button'); boton.textContent = 'Hacer foto';
          const video = document.createElement('video'); video.style.display = 'block';
          const stream = await navigator.mediaDevices.getUserMedia({video: true});
          document.body.appendChild(div); div.appendChild(video); div.appendChild(boton);
          video.srcObject = stream; await video.play();
          google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
          await new Promise((resolve) => boton.onclick = resolve);
          const canvas = document.createElement('canvas');
          canvas.width = video.videoWidth; canvas.height = video.videoHeight;
          canvas.getContext('2d').drawImage(video, 0, 0);
          stream.getVideoTracks()[0].stop(); div.remove();
          return canvas.toDataURL('image/jpeg', 0.9);
        }"""))
        datos = eval_js('hacerFoto()')
        open(fichero, 'wb').write(b64decode(datos.split(',')[1]))
    else:
        input('Colócate delante de la webcam y pulsa Enter para hacer la foto...')
        cam = cv2.VideoCapture(0)
        ok, frame = False, None
        for _ in range(15):                      # unos fotogramas para que la cámara ajuste la luz
            ok, frame = cam.read()
        cam.release()
        if not ok:
            raise RuntimeError('No he podido abrir la webcam. Usa subir_foto("ruta/a/tu_foto.jpg").')
        cv2.imwrite(fichero, frame)
    return cv2.cvtColor(cv2.imread(fichero), cv2.COLOR_BGR2RGB)

def subir_fichero():
    """Colab: abre el diálogo de subida. VS Code / Jupyter local: pide la ruta del fichero."""
    if EN_COLAB:
        from google.colab import files
        return list(files.upload())[0]
    return input('Ruta del fichero: ').strip().strip('"')

def subir_foto(ruta=None):
    ruta = ruta or subir_fichero()
    return cv2.cvtColor(cv2.imread(ruta), cv2.COLOR_BGR2RGB)

opciones = vision.PoseLandmarkerOptions(base_options=mpt.BaseOptions(model_asset_path='pose.task'),
                                        running_mode=vision.RunningMode.IMAGE)
def pose_de(imagen):
    """Devuelve los 33 puntos de la primera persona (o None si no hay nadie)."""
    with vision.PoseLandmarker.create_from_options(opciones) as detector:
        r = detector.detect(a_mp(imagen))
    return r.pose_landmarks[0] if r.pose_landmarks else None

## Nivel 1 · ¿Manos arriba?
Hazte una foto con la webcam (o sube una con `subir_foto()`) con las manos arriba, y otra con las manos abajo.
Completa la función: devuelve `True` si **las dos muñecas** están por encima de la **nariz**.

In [ ]:
def manos_arriba(persona):
    nariz = persona[P.NOSE]
    izq, der = persona[P.LEFT_WRIST], persona[P.RIGHT_WRIST]
    return None   # TODO: True si las dos muñecas tienen una y MENOR que la de la nariz

foto = foto_webcam()          # o: foto = subir_foto()
persona = pose_de(foto)
if persona is None:
    print('No veo a nadie: aléjate un poco de la cámara.')
else:
    texto = '¡Manos arriba!' if manos_arriba(persona) else 'Manos abajo'
    mostrar(dibujar_esqueleto(foto, [persona], CONEXIONES_POSE), texto)

## Nivel 2 · Abajo, en cruz o arriba
El ángulo del **hombro** es el que forman cadera → hombro → codo:
cerca de 0° el brazo está pegado al cuerpo, cerca de 90° en cruz y cerca de 180° arriba.
Completa `clasificar_brazo` con estos cortes: menos de 45° → `'abajo'`, de 45° a 135° → `'en cruz'`, más de 135° → `'arriba'`.

In [ ]:
def clasificar_brazo(persona, lado, W, H):
    a = angulo(persona[P[f'{lado}_HIP']], persona[P[f'{lado}_SHOULDER']], persona[P[f'{lado}_ELBOW']], W, H)
    # TODO: devuelve 'abajo', 'en cruz' o 'arriba' según el ángulo a
    return None, a

foto = foto_webcam()          # prueba un brazo en cruz y el otro arriba
persona = pose_de(foto)
H, W = foto.shape[:2]
izq, a_izq = clasificar_brazo(persona, 'LEFT', W, H)
der, a_der = clasificar_brazo(persona, 'RIGHT', W, H)
mostrar(dibujar_esqueleto(foto, [persona], CONEXIONES_POSE),
        f'Brazo izquierdo: {izq} ({a_izq:.0f}°) · Brazo derecho: {der} ({a_der:.0f}°)')

## Nivel 3 · Contador de sentadillas
Graba con el móvil 10–20 s haciendo sentadillas **de lado a la cámara** y súbelo.
El bucle que lee el vídeo ya está hecho; tú completas la **máquina de estados**:
- si estás `'arriba'` y el ángulo de la rodilla baja de `ABAJO`, pasas a `'abajo'`;
- si estás `'abajo'` y el ángulo sube de `ARRIBA`, vuelves a `'arriba'` y sumas una repetición.

In [ ]:
ABAJO, ARRIBA = 100, 160

def actualizar(estado, reps, a):
    # TODO: implementa las dos reglas y devuelve (estado, reps)
    return estado, reps

video = subir_fichero()            # Colab: sube el vídeo · local: escribe su ruta
op_video = vision.PoseLandmarkerOptions(base_options=mpt.BaseOptions(model_asset_path='pose.task'),
                                        running_mode=vision.RunningMode.VIDEO)
cap = cv2.VideoCapture(video); fps = cap.get(cv2.CAP_PROP_FPS) or 30
estado, reps, angulos, n = 'arriba', 0, [], 0
with vision.PoseLandmarker.create_from_options(op_video) as detector:
    while True:
        ok, frame = cap.read()
        if not ok: break
        h, w = frame.shape[:2]
        r = detector.detect_for_video(a_mp(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)), int(n * 1000 / fps)); n += 1
        if r.pose_landmarks:
            q = r.pose_landmarks[0]
            lado = 'LEFT' if q[P.LEFT_KNEE].visibility > q[P.RIGHT_KNEE].visibility else 'RIGHT'
            a = angulo(q[P[f'{lado}_HIP']], q[P[f'{lado}_KNEE']], q[P[f'{lado}_ANKLE']], w, h)
            angulos.append(a)
            estado, reps = actualizar(estado, reps, a)
cap.release()
print('Sentadillas:', reps)
plt.figure(figsize=(12, 3)); plt.plot(angulos)
plt.axhline(ABAJO, ls='--', c='tab:orange'); plt.axhline(ARRIBA, ls='--', c='tab:blue')
plt.ylabel('ángulo rodilla (°)'); plt.xlabel('fotograma'); plt.show()

## Si te sobra tiempo
- Cambia el nivel 3 para contar **flexiones** o **curl de bíceps** con el ángulo del codo.
- Combina niveles: un «Simón dice» que pida una postura (brazos en cruz, manos arriba…) y compruebe si la haces.